# Scheduling observations by expected information gain

This notebook builds photomancy's value-of-information loop from the ground up: it opens with
the theory of how an observation is scored, then puts that score in a closed loop that decides
what to observe next. The demonstration is a radial-velocity period search, the setting where
information-driven scheduling has the most to gain over a fixed cadence, but the loop itself is
generic, the same `Posterior`, `cluster_to_mixture`, and `eig_linearized` that drive scheduling at
every scale.

## The value of an observation

An observation is worth taking to the extent that it is expected to teach us something.
photomancy makes that precise with the expected information gain, the mutual information between
the parameters $z$ and a candidate observation $Y$,

$$ \mathrm{EIG}(Y) = I(z; Y) = \mathbb{E}_{Y}\big[\, D_{\mathrm{KL}}\!\big(p(z \mid Y) \,\|\, p(z)\big) \,\big], $$

equivalently the expected reduction in the entropy of the belief once the data arrive. A
scheduler that maximizes this quantity points where the next measurement is expected to sharpen
the posterior the most. The [mathematical foundations](../explanation/mathematical-foundations)
derive the terms used below.

## Two ways an observation informs

For a belief held as a Gaussian mixture, the gain splits into two readable pieces. Within a
single mode, linearizing the forward model about the mode, $y = J z + \varepsilon$ with Jacobian
$J = \partial y / \partial z$ and measurement covariance $R$, the matrix $J^\top R^{-1} J$ is
the Fisher information the observation carries about $z$ (the variable named `fim` in
`photomancy.infer.eig`). The posterior precision updates by adding it to the prior precision,
$\Sigma_{\mathrm{new}}^{-1} = \Sigma_{\mathrm{old}}^{-1} + J^\top R^{-1} J$, and the information
gained is the log-volume contraction of the covariance,

$$ \mathrm{EIG}_{\mathrm{geom}} = \tfrac{1}{2}\big(\log|\Sigma_{\mathrm{old}}| - \log|\Sigma_{\mathrm{new}}|\big). $$

Maximizing this term over candidates is Bayesian D-optimal design. Across modes, an
observation helps instead by telling them apart. With per-mode predictions $y_k$, weights
$w_k$, and full per-mode predictive covariances $S_k = J_k \Sigma_k J_k^\top + R$, the
information about which mode is correct has no closed form, so the alias-breaking term is an
upper bound on it, the Gaussian moment-matched entropy bound capped at the mode entropy
$H(w)$,

$$ \mathrm{EIG}_{\mathrm{alias}} = \min\Big\{ \tfrac{1}{2}\Big[\log|S_{\mathrm{mix}}| - \sum_k w_k \log|S_k|\Big],\; H(w) \Big\}, $$

where $S_{\mathrm{mix}}$ is the covariance of the whole mixture predictive, including the
spread of the $y_k$. The first term is the Fisher-information gain that refines a
well-isolated estimate, the second resolves which of several competing explanations is correct
and is the part the Fisher information cannot describe, since it is a property of the discrete
spread across modes. `photomancy.infer.eig_linearized` evaluates both over a batch of
candidates against a mixture posterior, returning their sum as an approximate score and each
part.

## The scheduling loop

A myopic scheduler chains the gain into a loop. Given the current belief, it scores the
candidate observations, takes the most informative one, folds the result into the belief, and
repeats:

1. score every candidate with the expected information gain,
2. observe at the candidate that maximizes it,
3. update the belief by refitting with the data so far, warm-started from the current modes,
4. repeat.

The problem below is built so that the two gain terms matter in turn. A radial-velocity signal
is sampled on a nightly cadence, which leaves the period aliased, so the belief starts multimodal
and the early gain is alias-breaking. Once the alias is resolved, the belief is a single mode and
the gain becomes geometric, refining the period.

In [ ]:
import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from priorities import Uniform

from photomancy import viz
from photomancy.infer import (
    Fit,
    ForwardTerm,
    Laplace,
    cluster_to_mixture,
    eig_linearized,
    ravel_samples,
    run,
)
from photomancy.infer.likelihoods import gaussian_loglike

jax.config.update("jax_enable_x64", True)

PI = jnp.pi


class RV(eqx.Module):
    # the scene: a radial-velocity signal's period (as log P), amplitude, and phase
    logP: jnp.ndarray
    K: jnp.ndarray
    phi: jnp.ndarray


def rv(p, t):
    return p.K * jnp.sin(2 * PI * t / 10.0**p.logP + p.phi)


class RVForward(eqx.Module):
    # the forward model: the signal at the observation times
    t: jnp.ndarray

    def __call__(self, p):
        return rv(p, self.t)


P_true = 2.7
f_true = 1.0 / P_true
# integer-day sampling aliases a period with those at f_true + n cycles/day
alias_P = np.array([P_true, 1.0 / (f_true + 1.0), 1.0 / (f_true + 2.0)])
truth = RV(logP=jnp.array(np.log10(P_true)), K=jnp.array(5.0), phi=jnp.array(0.7))
err = 0.4  # m/s

t_init = jnp.arange(0.0, 8.0)  # eight nightly epochs
y_init = rv(truth, t_init) + err * jax.random.normal(jax.random.key(0), t_init.shape)

# one free law per entry of the scene
laws = {
    "logP": Uniform(low=jnp.array(np.log10(0.3)), high=jnp.array(np.log10(5.0))),
    "K": Uniform(low=jnp.array(0.0), high=jnp.array(10.0)),
    "phi": Uniform(low=jnp.array(-PI), high=jnp.array(PI)),
}
# one explicit start per alias, as a dict with a leading axis of length n_modes
inits = {
    "logP": jnp.array(np.log10(alias_P)),
    "K": jnp.full(3, 4.0),
    "phi": jnp.full(3, 0.5),
}

The mixture and the gain calculation work on the flat vector of the free entries, which
`ravel_samples` builds in sorted key order and can unravel again, so no entry is placed by
hand. The helpers below build a fit from the data so far, summarize its cloud as a mixture
with one component per kept mode (`Laplace` already merges starts that
converge to the same mode, and the modes it kept seed the clustering, so a mode that the
data have since ruled out stays as a component of weight near zero), read the starts for
the next fit from the component means, and give the forward model of a candidate observation
in the same flat coordinates.

In [ ]:
def make_fit(t_obs, y_obs):
    base = RV(logP=jnp.array(0.0), K=jnp.array(0.0), phi=jnp.array(0.0))
    term = ForwardTerm(RVForward(t_obs), gaussian_loglike(y_obs, err))
    return Fit(base, laws, (term,))


def fit_modes(t_obs, y_obs, starts, key):
    # one Laplace mode per start; the cloud is the posterior
    fit = make_fit(t_obs, y_obs)
    backend = Laplace(n_draws=4000, n_modes=len(starts["K"]), n_steps=400, init=starts)
    return run(fit, backend, key)


def to_mixture(post, key):
    # one component per kept Laplace mode, seeded with the modes themselves
    info = post.info
    kept = np.asarray(info.kept)
    modes = jax.tree.map(lambda a: a[kept], info.modes)
    return cluster_to_mixture(post, int(info.n_modes_kept), key, init=modes)


def next_starts(mix):
    # the component means, as the starts of the next fit
    return unravel(mix.component_distribution.loc)


def eig_forward(z, t):
    # z is the flat vector of the fit
    p = unravel(z)
    return jnp.atleast_1d(p["K"] * jnp.sin(2 * PI * t / 10.0 ** p["logP"] + p["phi"]))


def periods_weights(mix):
    # the period and weight of each component
    P = 10.0 ** np.array(next_starts(mix)["logP"])
    return P, np.array(jax.nn.softmax(mix.mixing_logits))


def true_weight(mix):
    P, w = periods_weights(mix)
    return float(w[np.argmin(np.abs(P - P_true))])


def observe(t_obs, y_obs, t_star, key):
    y_star = rv(truth, jnp.asarray(t_star)) + err * jax.random.normal(key, ())
    return (
        jnp.concatenate([t_obs, jnp.atleast_1d(jnp.asarray(t_star))]),
        jnp.concatenate([y_obs, jnp.atleast_1d(y_star)]),
    )

## A problem with aliases

The signal is a single sinusoid, a planet's radial-velocity wobble
$v(t) = K \sin(2\pi t / P + \varphi)$, observed on eight consecutive nights. Because the samples
fall on integer days, the true period and the shorter periods whose frequencies differ from
$1/P$ by a whole number of cycles per day reproduce the same eight values, so the data alone
cannot tell them apart.

In [ ]:
tt = np.linspace(-0.5, 8.0, 600)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(tt, np.array(rv(truth, jnp.array(tt))), color="0.7", lw=1, label="true signal")
ax.scatter(
    np.array(t_init),
    np.array(y_init),
    c="crimson",
    s=35,
    zorder=3,
    label="nightly data",
)
ax.set_xlabel("time [days]")
ax.set_ylabel("radial velocity [m/s]")
ax.set_title("eight nightly radial-velocity measurements")
ax.legend()
plt.show()

## The multimodal belief

A multi-start Laplace fit, `Laplace(n_modes=3, init=starts)` with one start per candidate
period, returns a cloud of draws weighted by the evidence of each mode, and
`cluster_to_mixture` summarizes it as a mixture. The three modes are the true period and its
two nightly aliases, and the true period carries the most weight (0.70) but the two aliases
together keep about 30 percent of it, because they fit the nightly data almost as well. Plotted as curves they are nearly indistinguishable at the observed nights
and disagree everywhere between them, which is exactly the structure an observation can
exploit.

In [ ]:
post0 = fit_modes(t_init, y_init, inits, jax.random.key(10))
_, unravel = ravel_samples(post0)  # the flat order of every fit below
mix0 = to_mixture(post0, jax.random.key(11))
P, w = periods_weights(mix0)
for i in np.argsort(-w):
    print(f"mode: P = {P[i]:.3f} d   weight = {w[i]:.3f}")

tt = np.linspace(0.0, 8.0, 800)
loc0 = jax.tree.map(np.asarray, next_starts(mix0))
fig, ax = plt.subplots(figsize=(7, 3.2))
for i in np.argsort(-w):
    K, logP, phi = loc0["K"][i], loc0["logP"][i], loc0["phi"][i]
    curve = K * np.sin(2 * np.pi * tt / 10.0**logP + phi)
    ax.plot(tt, curve, lw=1.3, label=f"P = {P[i]:.2f} d  (w = {w[i]:.2f})")
ax.scatter(np.array(t_init), np.array(y_init), c="crimson", s=35, zorder=3)
ax.set_xlabel("time [days]")
ax.set_ylabel("radial velocity [m/s]")
ax.set_title("three period modes: agree at the nights, disagree between")
ax.legend(fontsize=8)
plt.show()

## Scoring candidate observations

The candidates are the next ten days after the last night, the horizon the loop below uses. The
alias-breaking term reaches its cap, the mode entropy $H(w) = 0.80$ nats, at many half-night
epochs, and is about 0.4 nats at the integer days, where the aliases agree only approximately
because their fitted periods differ slightly. The Fisher-information term is larger (up to
1.9 nats) and varies with the epoch just as irregularly, so the best epoch, $t^\star = 16.4$ d
with 2.4 nats in total, is an off-cadence time that scores well on both terms rather than the
next night of the nightly run.

In [ ]:
t_last = float(jnp.max(t_init))
cand = jnp.linspace(t_last + 0.3, t_last + 10.0, 200)  # the horizon the loop uses
e = eig_linearized(mix0, cand, eig_forward, err**2)
t_star = float(cand[int(jnp.nanargmax(e.total))])

result = viz.plot_eig(cand, e, epochs_d=t_init)
result.fig.set_size_inches(7, 3.4)
result.ax.set_title("expected information gain over candidate times")
plt.show()
print(f"best next time t* = {t_star:.2f} d  ({float(jnp.max(e.total)):.2f} nats)")

## One observation breaks the alias

Taking the single most informative epoch and folding it into the belief collapses the mixture.
The refit is warm-started from the means of the current mixture, one start per mode. The true
period keeps essentially all of the weight, and the aliases, which predicted the wrong
velocity at that off-cadence time, are ruled out.

In [ ]:
t1, y1 = observe(t_init, y_init, t_star, jax.random.key(7))
post1 = fit_modes(t1, y1, next_starts(mix0), jax.random.key(12))
mix1 = to_mixture(post1, jax.random.key(13))
P1, w1 = periods_weights(mix1)

x = np.arange(len(P))
# the weight each period mode keeps: the components of the new mixture near its period
w1_by = np.array([w1[np.abs(P1 - p) < 0.03 * p].sum() for p in P])
w0_by = w
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(x - 0.18, w0_by, width=0.36, label="before", color="0.7")
ax.bar(x + 0.18, w1_by, width=0.36, label="after one epoch", color="crimson")
ax.set_xticks(x)
ax.set_xticklabels([f"{p:.2f} d" for p in P])
ax.set_xlabel("period mode")
ax.set_ylabel("posterior weight")
ax.set_title(f"one observation at t* = {t_star:.1f} d collapses the alias")
ax.legend()
plt.show()

## The full loop versus a fixed cadence

Run end to end, the information-driven schedule puts essentially all of the weight on the true
period after its first extra epoch and then extends the baseline to sharpen the period. A
fixed nightly cadence, the natural default, never resolves the alias, because every
integer-day sample leaves the periods degenerate: about 30 percent of the weight stays on the
aliases however many nights are added. The information-driven loop also has the tighter
period at every step: $\sigma(\log P)$ is about 0.001 against 0.003 to 0.004 for the fixed
cadence, and the gap does not close by the sixth observation.

In [ ]:
i_logP = int(unravel(jnp.arange(3.0))["logP"])  # where log P sits in the flat vector


def run_loop(strategy, n_steps=6):
    t_obs, y_obs = t_init, y_init
    post = fit_modes(t_obs, y_obs, inits, jax.random.key(20))
    mix = to_mixture(post, jax.random.key(21))
    rec = []
    for step in range(n_steps):
        if strategy == "greedy":
            t_last = float(jnp.max(t_obs))
            c = jnp.linspace(t_last + 0.3, t_last + 10.0, 200)  # a forward horizon
            ev = eig_linearized(mix, c, eig_forward, err**2)
            t_star = c[int(jnp.nanargmax(ev.total))]
        else:
            t_star = jnp.array(8.0 + step * 3.0)  # keep adding nights
        t_obs, y_obs = observe(t_obs, y_obs, t_star, jax.random.key(100 + step))
        post = fit_modes(t_obs, y_obs, next_starts(mix), jax.random.key(200 + step))
        mix = to_mixture(post, jax.random.key(300 + step))
        _, w_now = periods_weights(mix)
        best = int(np.argmax(w_now))
        tril = mix.component_distribution.scale_tril[best]
        var = (tril @ tril.T)[i_logP, i_logP]
        rec.append((true_weight(mix), float(jnp.sqrt(var))))
    return np.array(rec)


g, u = run_loop("greedy"), run_loop("uniform")
steps = np.arange(1, len(g) + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.5, 3.4))
a1.plot(steps, g[:, 0], "o-", color="navy", label="information-driven")
a1.plot(steps, u[:, 0], "s--", color="0.55", label="fixed cadence")
a1.set_xlabel("extra observations")
a1.set_ylabel("weight on the true period")
a1.set_ylim(0, 1.05)
a1.legend(fontsize=8)
a2.plot(steps, g[:, 1], "o-", color="navy", label="information-driven")
a2.plot(steps, u[:, 1], "s--", color="0.55", label="fixed cadence")
a2.set_xlabel("extra observations")
a2.set_ylabel("period uncertainty, sigma(log P)")
a2.set_yscale("log")
a2.legend(fontsize=8)
fig.suptitle("information-driven scheduling resolves the alias and sharpens the period")
fig.tight_layout()
plt.show()

## What this is

This loop is the photomancy scheduler in miniature. The belief is a `Posterior`, the candidate
score is `eig_linearized` on its `cluster_to_mixture` summary, and the update is a refit
warm-started from the mixture's means, the same three pieces that drive scheduling at every
scale. Swapping the radial-velocity forward model for an orbit, an atmosphere, or a
coronagraph image reuses the loop unchanged, which is the point of the
[architecture](../explanation/architecture): one information currency, computed the same way,
from a single exposure up to a whole mission. The orbit layer ships its own
`eig_astrometry` for exactly this, and the detectability term extends the same idea to
the question of whether a planet is there to be seen at all. The
[abstract walkthrough](walkthrough) shows the same machinery on a simpler spatial problem.